# MCP Transport

**Module:** 12-mcp

**Notebook:** `06-mcp-transport.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Transport Role** with clear contracts and failure modes
- Explain and apply **stdio Transport** with clear contracts and failure modes
- Explain and apply **Remote Transports** with clear contracts and failure modes
- Explain and apply **Message Framing Sketch** with clear contracts and failure modes
- Explain and apply **Timeouts & Cancellation** with clear contracts and failure modes
- Explain and apply **Local vs Remote Trade-offs** with clear contracts and failure modes
- Explain and apply **Connectivity Health Checks** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — MCP Transport

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Transport Role**
2. **stdio Transport**
3. **Remote Transports**
4. **Message Framing Sketch**
5. **Timeouts & Cancellation**
6. **Local vs Remote Trade-offs**
7. **Connectivity Health Checks**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Transport Role

### Definition
**Transport Role** is a core building block in 06-mcp-transport within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Transport Role typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Transport Role: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Transport Role as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Transport Role as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Transport Role
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Transport Role when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Transport Role improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Transport Role" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Transport Role"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


## stdio Transport

### Definition
**stdio Transport** is a core building block in 06-mcp-transport within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around stdio Transport typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For stdio Transport: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain stdio Transport as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating stdio Transport as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for stdio Transport
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use stdio Transport when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "stdio Transport" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "stdio Transport"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


### Worked scenario — stdio Transport

**Situation:** A team wants to productionize a feature involving **stdio Transport**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Remote Transports

### Definition
**Remote Transports** is a core building block in 06-mcp-transport within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Remote Transports typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Remote Transports: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Remote Transports as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Remote Transports as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Remote Transports
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Remote Transports when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Remote Transports" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Remote Transports"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


## Message Framing Sketch

### Definition
**Message Framing Sketch** is a core building block in 06-mcp-transport within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Message Framing Sketch typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Message Framing Sketch: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Message Framing Sketch as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Message Framing Sketch as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Message Framing Sketch
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Message Framing Sketch when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Message Framing Sketch" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Message Framing Sketch"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Message Framing Sketch"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Message Framing Sketch"}
strong = {"definition": "Message Framing Sketch", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Message Framing Sketch"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Message Framing Sketch", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Message Framing Sketch

**Situation:** A team wants to productionize a feature involving **Message Framing Sketch**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Timeouts & Cancellation

### Definition
**Timeouts & Cancellation** is a core building block in 06-mcp-transport within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Timeouts & Cancellation typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Timeouts & Cancellation: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Timeouts & Cancellation as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Timeouts & Cancellation as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Timeouts & Cancellation
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Timeouts & Cancellation when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Timeouts & Cancellation" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Timeouts & Cancellation"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Timeouts & Cancellation"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Timeouts & Cancellation"}
strong = {"definition": "Timeouts & Cancellation", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Timeouts & Cancellation"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Timeouts & Cancellation", "passed": len(checks)-len(failed), "failed": failed})


## Local vs Remote Trade-offs

### Definition
**Local vs Remote Trade-offs** helps you choose among alternatives using explicit criteria rather than hype.

### Why it matters
In Model Context Protocol (MCP), weak designs around Local vs Remote Trade-offs typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
List options, define criteria (quality, cost, latency, ops, lock-in), score with evidence, document the decision.

### Intuition
Explain Local vs Remote Trade-offs as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Local vs Remote Trade-offs as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Local vs Remote Trade-offs
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Local vs Remote Trade-offs when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Local vs Remote Trade-offs" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Local vs Remote Trade-offs"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Local vs Remote Trade-offs"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Local vs Remote Trade-offs"}
strong = {"definition": "Local vs Remote Trade-offs", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Local vs Remote Trade-offs"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Local vs Remote Trade-offs", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Local vs Remote Trade-offs

**Situation:** A team wants to productionize a feature involving **Local vs Remote Trade-offs**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Connectivity Health Checks

### Definition
**Connectivity Health Checks** is a core building block in 06-mcp-transport within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Connectivity Health Checks typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Connectivity Health Checks: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Connectivity Health Checks as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Connectivity Health Checks as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Connectivity Health Checks
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Connectivity Health Checks when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Connectivity Health Checks" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Connectivity Health Checks"
    notebook: str = "06-mcp-transport"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Connectivity Health Checks"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Connectivity Health Checks"}
strong = {"definition": "Connectivity Health Checks", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Connectivity Health Checks"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Connectivity Health Checks", "passed": len(checks)-len(failed), "failed": failed})


## Comparison Snapshot

Use this table when reviewing designs in **MCP Transport**.

| Topic | Do | Don't |
|-------|----|-------|
| Transport Role | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| stdio Transport | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Remote Transports | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Message Framing Sketch | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Timeouts & Cancellation | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Local vs Remote Trade-offs | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Transport Role | Key concept covered in this notebook; see its section for definition and pitfalls |
| stdio Transport | Key concept covered in this notebook; see its section for definition and pitfalls |
| Remote Transports | Key concept covered in this notebook; see its section for definition and pitfalls |
| Message Framing Sketch | Key concept covered in this notebook; see its section for definition and pitfalls |
| Timeouts & Cancellation | Key concept covered in this notebook; see its section for definition and pitfalls |
| Local vs Remote Trade-offs | Key concept covered in this notebook; see its section for definition and pitfalls |
| Connectivity Health Checks | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **MCP Transport** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **12-mcp**.


## Try It Yourself

1. Implement a failing test/fixture for **Transport Role**, then fix your demo until it passes.
2. Implement a failing test/fixture for **stdio Transport**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Remote Transports**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Message Framing Sketch**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Timeouts & Cancellation**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
